In [77]:
import pandas as pd
import re
import json

In [78]:
df = pd.read_csv('../data/processed/cluster_categorized_data.csv', index_col=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print('\nNull counts:')
print(df.isnull().sum())

Shape: (3689, 18)
Columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression', 'filename', 'input_text', 'text_source', 'structured', 'summary', 'normal', 'conditions', 'findings_extracted', 'cluster', 'final_label']

Null counts:
uid                      0
MeSH                     0
Problems                 0
image                    0
indication              83
comparison            1121
findings               490
impression              29
filename                 0
input_text              23
text_source              0
structured               0
summary                 23
normal                   0
conditions            2633
findings_extracted    3689
cluster                  0
final_label              0
dtype: int64


In [79]:
df.head(3)

,uid,MeSH,Problems,image,indication,comparison,findings,impression,filename,input_text,text_source,structured,summary,normal,conditions,findings_extracted,cluster,final_label
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png,Normal chest x-XXXX.,impression,"{'normal': True, 'conditions': [], 'findings':...",Normal chest x-XXXX.,True,NaN,NaN,1,Pulmonary Disease
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png,No acute pulmonary findings.,impression,"{'normal': True, 'conditions': [], 'findings':...",No acute pulmonary findings.,True,NaN,NaN,3,Normal (Explicit)
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p...",3_IM-1384-1001.dcm.png,"No displaced rib fractures, pneumothorax, or p...",impression,"{'normal': False, 'conditions': ['pleural effu...","No displaced rib fractures, pneumothorax, or p...",False,"pleural effusion, pneumothorax, fracture",NaN,5,Cardiac / Structural


In [80]:
# Audit the fields we will use before touching anything
fields = ['impression', 'findings', 'indication', 'MeSH']
for col in fields:
    null_n = df[col].isna().sum()
    pct    = null_n / len(df) * 100
    sample = str(df[col].dropna().iloc[0])[:80] if df[col].notna().any() else 'ALL NULL'
    print(f'{col:<12}  null={null_n} ({pct:.1f}%)  sample: {sample}')

impression    null=29 (0.8%)  sample: Normal chest x-XXXX.
findings      null=490 (13.3%)  sample: The cardiac silhouette and mediastinum size are within normal limits. There is n
indication    null=83 (2.2%)  sample: Positive TB test
MeSH          null=0 (0.0%)  sample: normal


In [81]:
import re

In [ ]:
# prevents touching 'x' inside real words (examination, thorax, complex, etc.)
_ANON_TOKEN = re.compile(r'(?<![a-zA-Z])[Xx]{2,}(?![a-zA-Z])')

# Matches hyphenated anon compounds: 'x-XXXX', 'XXXX-XXXX'
_ANON_HYPHEN = re.compile(r'\b[Xx]{1,}-[Xx]{2,}\b|\b[Xx]{2,}-[Xx]{1,}\b')

# Numbers with optional space + unit: '1.5 cm', '3mm', '14%' → ';'
_NUMBER = re.compile(r'\b\d+(?:\.\d+)?\s*(?:mm|cm|ml|kg|mg|%)?\b')

# Dangling punct mid-sentence after token removal: 'sternotomy . Enlarged'
_DANGLE_MID = re.compile(r'\s+[.,;:]\s+(?=[A-Z])')

# Dangling punct at end of string: 'Inferior .' → 'Inferior'
_DANGLE_END = re.compile(r'\s+[.,;:]\s*$')

# Two or more consecutive spaces
_MULTI_SPACE = re.compile(r' {2,}')

# Repeated punctuation: '..' ',,,' → single char
_REPEAT_PUNCT = re.compile(r'([.,;:]){2,}')


def clean_radiology_text(text: str) -> str:
    """
    Comprehensive cleaner for IU chest X-ray radiology report text.

    """
    # ── Guard: preserve NaN/None as-is 
    if text is None or (isinstance(text, float)):
        return text
    text = str(text).strip()
    if not text:
        return text

    # remove hyphenated anon compounds before bare tokens 

    text = _ANON_HYPHEN.sub('', text)

    # remove bare XXXX tokens
    text = _ANON_TOKEN.sub('', text)

    # replace numbers and measurements with ';'
    text = _NUMBER.sub(';', text)

    # repair dangling punctuation left by token removal 
    text = _DANGLE_MID.sub('. ', text)   # 'sternotomy . Enlarged' → 'sternotomy. Enlarged'
    text = _DANGLE_END.sub('', text)     # 'Inferior .'            → 'Inferior'

    # collapse multi-spaces 
    text = _MULTI_SPACE.sub(' ', text)

    # normalise repeated punctuation '...' → '.' 
    text = _REPEAT_PUNCT.sub(r'\1', text)

    # strip leading/trailing non-word chars
    text = re.sub(r'^[^\w]+', '', text)
    text = re.sub(r'[^\w.]+$', '', text)

    return text.strip()


In [ ]:
# Clean every text field that feeds into the embedding
for col in ['impression', 'findings', 'indication']:
    df[col] = df[col].apply(clean_radiology_text)

# MeSH has a different structure (path/semicolon separated) — dedicated normaliser
def clean_mesh(text):
    """
    Normalise MeSH terms for embedding.
    
    """
    if text is None or isinstance(text, float):
        return None
    text = str(text).strip()
    if not text or text.lower() == 'normal':
        return None
    text = text.replace('/', ' ').replace(';', ', ')
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

df['MeSH_clean'] = df['MeSH'].apply(clean_mesh)

print('Cleaning complete.')
print(f"MeSH_clean null (normal-only or missing): {df['MeSH_clean'].isna().sum()}")

Cleaning complete.
MeSH_clean null (normal-only or missing): 1343


In [84]:
df[['impression','findings','indication','MeSH_clean']].head(5)

,impression,findings,indication,MeSH_clean
0,Normal chest,The cardiac silhouette and mediastinum size ar...,Positive TB test,NaN
1,No acute pulmonary findings.,Borderline cardiomegaly. Midline sternotomy. E...,Preop bariatric surgery.,"Cardiomegaly borderline, Pulmonary Artery enla..."
2,"No displaced rib fractures, pneumothorax, or p...",NaN,"rib pain after a , steps this. Pain to R back,...",NaN
3,Bullous emphysema and interstitial fibrosis. ....,There are diffuse bilateral interstitial and a...,year-old with,"Pulmonary Disease, Chronic Obstructive, Bullou..."
4,No acute cardiopulmonary abnormality.,The cardiomediastinal silhouette and pulmonary...,Chest and nasal congestion.,"Osteophyte thoracic vertebrae multiple small, ..."


In [85]:
# Verify XXXX removed from all cleaned text fields
for col in ['impression', 'findings', 'indication']:
    remaining = df[col].dropna().str.contains(r'[Xx]{2,}', regex=True).sum()
    print(f'XXXX remaining in {col:<12}: {remaining}')

XXXX remaining in impression  : 1
XXXX remaining in findings    : 0
XXXX remaining in indication  : 4


In [86]:
print('Null counts after cleaning:')
for col in ['impression','findings','indication','MeSH_clean']:
    print(f'  {col:<14}: {df[col].isna().sum()}')

Null counts after cleaning:
  impression    : 29
  findings      : 490
  indication    : 83
  MeSH_clean    : 1343


In [87]:
# Sample cleaned impression + MeSH pairs
for _, row in df.head(5).iterrows():
    print(f'impression : {row["impression"]}')
    print(f'indication : {row["indication"]}')
    print(f'MeSH_clean : {row["MeSH_clean"]}')
    print()

impression : Normal chest
indication : Positive TB test
MeSH_clean : nan

impression : No acute pulmonary findings.
indication : Preop bariatric surgery.
MeSH_clean : Cardiomegaly borderline, Pulmonary Artery enlarged

impression : No displaced rib fractures, pneumothorax, or pleural effusion identified. Well-expanded and clear lungs. Mediastinal contour within normal limits. No acute cardiopulmonary abnormality identified.
indication : rib pain after a , steps this. Pain to R back, R elbow and R rib , no previous heart or lung hx, non-, no hx ca
MeSH_clean : nan

impression : Bullous emphysema and interstitial fibrosis. . Probably scarring in the left apex, although difficult to exclude a cavitary lesion. . Opacities in the bilateral upper lobes could represent scarring, however the absence of comparison exam, recommend short interval followup radiograph or CT thorax to document resolution.
indication : year-old with
MeSH_clean : Pulmonary Disease, Chronic Obstructive, Bullous Emphyse

In [88]:
before = len(df)

df = df[
    df['impression'].apply(_has_value) |
    df['findings'].apply(_has_value)   |
    df['indication'].apply(_has_value)
].reset_index(drop=True)

print(f"Dropped {before - len(df)} rows with no usable text fields.")
print(f"Remaining: {len(df)}")

Dropped 23 rows with no usable text fields.
Remaining: 3666


In [89]:
def _has_value(val) -> bool:
    """True only if val is a non-null, non-empty, non-literal-nan string."""
    if val is None:
        return False
    if isinstance(val, float):  # catches actual NaN
        return False
    s = str(val).strip()
    return bool(s) and s.lower() != 'nan'

In [ ]:
def build_embedding_text(row) -> str:
    parts = []

    indication = row.get('indication')
    if _has_value(indication):
        parts.append(f"Indication: {indication}")

    findings = row.get('findings')
    if _has_value(findings):
        parts.append(f"Findings: {findings}")

    impression = row.get('impression')
    if _has_value(impression):
        parts.append(f"Impression: {impression}")

    mesh = row.get('MeSH_clean')
    if _has_value(mesh):
        parts.append(f"Conditions: {mesh}")

    # ── Fallback chain for rows where all fields are null ─────────────────

    if not parts:
        raw_mesh = row.get('MeSH')
        if _has_value(raw_mesh) and str(raw_mesh).strip().lower() != 'normal':
            cleaned = str(raw_mesh).replace('/', ' ').replace(';', ', ')
            parts.append(f"Conditions: {cleaned}")
        else:
            # Last resort: label from clustering — at minimum the record
            # won't produce a zero/garbage embedding
            label = row.get('final_label', '')
            normal_flag = row.get('normal', '')
            parts.append(f"Label: {label if _has_value(label) else 'unknown'}, Normal: {normal_flag}")

    return ' | '.join(parts).strip()

In [91]:
df['embedding_text'] = df.apply(build_embedding_text, axis=1)

empty_mask = df['embedding_text'].str.strip() == ''
print(f'Rows with non-empty embedding_text: {(~empty_mask).sum()} / {len(df)}')
print(f'Rows with empty embedding_text:     {empty_mask.sum()}')
print()
print('Field coverage in assembled texts:')
for label, key in [('indication','Indication:'), ('findings','Findings:'),
                    ('impression','Impression:'), ('MeSH','Conditions:')]:
    count = df['embedding_text'].str.contains(key).sum()
    print(f'  {label:<12}: {count} / {len(df)} ({count/len(df)*100:.1f}%)')

Rows with non-empty embedding_text: 3666 / 3666
Rows with empty embedding_text:     0

Field coverage in assembled texts:
  indication  : 3389 / 3666 (92.4%)
  findings    : 3199 / 3666 (87.3%)
  impression  : 3660 / 3666 (99.8%)
  MeSH        : 2346 / 3666 (64.0%)


In [92]:
# empty_mask = [i for i, r in enumerate(check) if not r.get('text', '').strip()]
# assert not empty_mask, f'Empty text in {len(empty_mask)} records at indices: {empty_mask}'

In [93]:
# problem_indices = [15, 574, 833, 1063, 1068, 1073, 1202, 1206, 1426, 1456, 1500, 1570, 1634, 1650, 1956, 2398, 2480, 2543, 2646, 3095, 3157, 3487, 3663]

# for i in problem_indices:
#     row = df.iloc[i]
#     print(f"[{i}]")
#     print(f"  impression : {repr(row['impression'])}")
#     print(f"  findings   : {repr(row['findings'])}")
#     print(f"  indication : {repr(row['indication'])}")
#     print(f"  MeSH_clean : {repr(row['MeSH_clean'])}")
#     print()

In [94]:
print('Sample embedding texts:')
for i, row in df.head(5).iterrows():
    print(f'[{i}] {row["embedding_text"]}')
    print()

Sample embedding texts:
[0] Indication: Positive TB test | Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no of a pleural effusion. There is no evidence of pneumothorax. | Impression: Normal chest

[1] Indication: Preop bariatric surgery. | Findings: Borderline cardiomegaly. Midline sternotomy. Enlarged pulmonary arteries. Clear lungs. Inferior | Impression: No acute pulmonary findings. | Conditions: Cardiomegaly borderline, Pulmonary Artery enlarged

[2] Indication: rib pain after a , steps this. Pain to R back, R elbow and R rib , no previous heart or lung hx, non-, no hx ca | Impression: No displaced rib fractures, pneumothorax, or pleural effusion identified. Well-expanded and clear lungs. Mediastinal contour within normal limits. No acute cardiopulmonary abnormality identified.

[3] Indication: year-old with | Findings: There are diffuse bilateral interstitial and alveolar opac

In [95]:
json_data = []

for _, row in df.iterrows():
    image = row['filename'] if pd.notna(row.get('filename')) else None

    entry = {
        'id'     : row['uid'],
        'text'   : row['embedding_text'],
        'image'  : image,
        # Structured metadata kept alongside text so downstream consumers
        # can filter/group without re-parsing the embedding_text string
        'label'  : row.get('final_label'),
        'cluster': int(row['cluster']) if pd.notna(row.get('cluster')) else None,
        'normal' : bool(row['normal'])  if pd.notna(row.get('normal'))  else None,
    }
    json_data.append(entry)

print(f'Total records: {len(json_data)}')
print('Sample entry:')
print(json.dumps(json_data[0], indent=2))

Total records: 3666
Sample entry:
{
  "id": 1,
  "text": "Indication: Positive TB test | Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no of a pleural effusion. There is no evidence of pneumothorax. | Impression: Normal chest",
  "image": "1_IM-0001-4001.dcm.png",
  "label": "Pulmonary Disease",
  "cluster": 1,
  "normal": true
}


In [96]:
json_data[:3]

[{'id': 1,
  'text': 'Indication: Positive TB test | Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no of a pleural effusion. There is no evidence of pneumothorax. | Impression: Normal chest',
  'image': '1_IM-0001-4001.dcm.png',
  'label': 'Pulmonary Disease',
  'cluster': 1,
  'normal': True},
 {'id': 2,
  'text': 'Indication: Preop bariatric surgery. | Findings: Borderline cardiomegaly. Midline sternotomy. Enlarged pulmonary arteries. Clear lungs. Inferior | Impression: No acute pulmonary findings. | Conditions: Cardiomegaly borderline, Pulmonary Artery enlarged',
  'image': '2_IM-0652-1001.dcm.png',
  'label': 'Normal (Explicit)',
  'cluster': 3,
  'normal': True},
 {'id': 3,
  'text': 'Indication: rib pain after a , steps this. Pain to R back, R elbow and R rib , no previous heart or lung hx, non-, no hx ca | Impression: No displaced rib fractures, pneumothorax, or pleural effu

In [97]:
output_path = '../data/processed/embedding_input.json'

with open(output_path, 'w') as f:
    json.dump(json_data, f, indent=4)

# Round-trip verification
with open(output_path) as f:
    check = json.load(f)

assert len(check) == len(df), f'Row count mismatch: {len(check)} vs {len(df)}'
assert all('text' in r and r['text'] for r in check), 'Some records have empty text'

print(f'Saved {len(check)} records to {output_path}')
print('Verification: PASSED')
print('\nField presence across all records:')
for key in ['id','text','image','label','cluster','normal']:
    filled = sum(1 for r in check if r.get(key) is not None)
    print(f'  {key:<10}: {filled} / {len(check)}')

Saved 3666 records to ../data/processed/embedding_input.json
Verification: PASSED

Field presence across all records:
  id        : 3666 / 3666
  text      : 3666 / 3666
  image     : 3666 / 3666
  label     : 3666 / 3666
  cluster   : 3666 / 3666
  normal    : 3666 / 3666
